## **Import**

In [1]:
import pandas as pd

## **Load Dataset**

In [9]:
orders = pd.read_csv("data/olist_orders_dataset.csv")
customers = pd.read_csv("data/olist_customers_dataset.csv")
items = pd.read_csv("data/olist_order_items_dataset.csv")
products = pd.read_csv("data/olist_products_dataset.csv")
payments = pd.read_csv("data/olist_order_payments_dataset.csv")
translation = pd.read_csv("data/product_category_name_translation.csv")

## **Merge Orders + Customers**

In [10]:
df = pd.merge(orders, customers, on="customer_id", how="left")

## **Merge Order Items**

In [11]:
df = pd.merge(df, items, on="order_id", how="left")

## **Merge Products**

In [12]:
df = pd.merge(df, products, on="product_id", how="left")

## **Add Category Translation**

In [13]:
df = pd.merge(
    df,
    translation,
    on="product_category_name",
    how="left"
)

## **Merge Payments**

In [14]:
df = pd.merge(df, payments, on="order_id", how="left")

## **Clean Columns**

In [15]:
# Rename for clarity
df = df.rename(columns={
    "product_category_name_english": "product_category"
})

# Optional: drop original Portuguese column
#df = df.drop(columns=["product_category_name"])

## **Basic Validation**

In [16]:
df.shape

(118434, 31)

In [17]:
df.isnull().sum().head(10)

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 176
order_delivered_carrier_date     2074
order_delivered_customer_date    3397
order_estimated_delivery_date       0
customer_unique_id                  0
customer_zip_code_prefix            0
dtype: int64

# **ANALYSIS**

## **Revenue by State**

In [18]:
state_revenue = (
    df.groupby("customer_state")["price"]
    .sum()
    .reset_index()
    .sort_values(by="price", ascending=False)
)

state_revenue.head()

,customer_state,price
25,SP,5448301.72
18,RJ,1913564.16
10,MG,1639636.83
22,RS,787770.53
17,PR,705856.47


## **Top Product Categories**

In [19]:
top_categories = df["product_category"].value_counts().head(10)
top_categories

product_category
bed_bath_table           11823
health_beauty             9975
sports_leisure            8945
furniture_decor           8744
computers_accessories     8082
housewares                7355
watches_gifts             6201
telephony                 4721
garden_tools              4574
auto                      4379
Name: count, dtype: int64

## **Payment Type Distribution**

In [20]:
payment_dist = df["payment_type"].value_counts()
payment_dist

payment_type
credit_card    87286
boleto         23037
voucher         6407
debit_card      1698
not_defined        3
Name: count, dtype: int64

## **Top Customers by Spending**

In [21]:
top_customers = (
    df.groupby("customer_unique_id")["price"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_customers

customer_unique_id
0a0a92112bd4c708ca5fde585afaa872    13440.00
8af7ac63b2efbcbd88e5b11505e8098a    11383.95
c4b224d2c784bae11ae98b6ae9f2454c    10856.10
85963fd37bfd387aa6d915d8a1065486     9888.00
be74c431147c32ab2d7c7cef5e4a995f     9520.14
edf81e1f3070b9dac83ec83dacdbb9bc     7998.00
1033eef37a760a832f9b2332ce23fdc5     7799.40
58483a1c055dfb600f57c5b867174542     7798.00
da122df9eeddfedc1dc1f5349a1a690c     7388.00
763c8b1c9c68a0229c42c9fc6f662b93     7160.00
Name: price, dtype: float64

## **Average Order Value**

In [22]:
avg_order_value = df["price"].mean()
avg_order_value

np.float64(120.82284879765994)

## **Save Final Dataset**

In [23]:
df.to_csv("data/final_merged_data.csv", index=False)